In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/vision_lab"
SEQUENCE = "structure_texture"   # change this each of the 4 runs

SEQ_URLS = {
    "nostructure_notexture": "https://cvg.cit.tum.de/rgbd/dataset/freiburg3/rgbd_dataset_freiburg3_nostructure_notexture_near_withloop.tgz",
    "structure_notexture":   "https://cvg.cit.tum.de/rgbd/dataset/freiburg3/rgbd_dataset_freiburg3_structure_notexture_near.tgz",
    "nostructure_texture":   "https://cvg.cit.tum.de/rgbd/dataset/freiburg3/rgbd_dataset_freiburg3_nostructure_texture_near_withloop.tgz",
    "structure_texture":     "https://cvg.cit.tum.de/rgbd/dataset/freiburg3/rgbd_dataset_freiburg3_structure_texture_near.tgz",
}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls -la vision_workspace/inputs/
!ls -la vision_workspace/inputs/tum_sequence.tgz

ls: cannot access 'vision_workspace/inputs/': No such file or directory
ls: cannot access 'vision_workspace/inputs/tum_sequence.tgz': No such file or directory


In [ ]:
!rm -rf vision_workspace
import os
os.makedirs("vision_workspace/inputs", exist_ok=True)

!wget {SEQ_URLS[SEQUENCE]} -O vision_workspace/inputs/tum_sequence.tgz
!tar -xzf vision_workspace/inputs/tum_sequence.tgz -C vision_workspace/inputs

--2026-07-11 09:46:56--  https://cvg.cit.tum.de/rgbd/dataset/freiburg3/rgbd_dataset_freiburg3_structure_texture_near.tgz
Resolving cvg.cit.tum.de (cvg.cit.tum.de)... 131.159.19.110, 2a09:80c0:18::1110
Connecting to cvg.cit.tum.de (cvg.cit.tum.de)|131.159.19.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 567005874 (541M) [application/x-gzip]
Saving to: ‘vision_workspace/inputs/tum_sequence.tgz’

vision_workspace/in 100%[===================>] 540.74M  26.6MB/s    in 18s     

2026-07-11 09:47:15 (29.5 MB/s) - ‘vision_workspace/inputs/tum_sequence.tgz’ saved [567005874/567005874]



In [ ]:
extracted_root = [d for d in os.listdir("vision_workspace/inputs")
                   if os.path.isdir(f"vision_workspace/inputs/{d}")][0]
src = f"vision_workspace/inputs/{extracted_root}"

os.makedirs("vision_workspace/pipeline_v1_raw/rgb", exist_ok=True)
os.makedirs("vision_workspace/ground_truth_depth", exist_ok=True)

!mv {src}/rgb/* vision_workspace/pipeline_v1_raw/rgb/
!mv {src}/*.txt vision_workspace/pipeline_v1_raw/
!mv {src}/depth/* vision_workspace/ground_truth_depth/

print(f"RGB frames: {len(os.listdir('vision_workspace/pipeline_v1_raw/rgb'))}")
print(f"Depth frames: {len(os.listdir('vision_workspace/ground_truth_depth'))}")

RGB frames: 1099
Depth frames: 1065


In [ ]:
def tum_text(text_path):
    log_dict = {}
    with open(text_path, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.strip().split()
            if len(parts) == 2:
                log_dict[float(parts[0])] = parts[1]
    return log_dict

rgb_registry = tum_text("vision_workspace/pipeline_v1_raw/rgb.txt")
depth_registry = tum_text("vision_workspace/pipeline_v1_raw/depth.txt")

In [ ]:
matched_pairs = []
max_allowed_gap = 0.02

for depth_time, depth_rel_path in depth_registry.items():
    closest_rgb_time = min(rgb_registry.keys(), key=lambda t: abs(t - depth_time))
    if abs(closest_rgb_time - depth_time) <= max_allowed_gap:
        matched_pairs.append({
            'rgb_filename': os.path.basename(rgb_registry[closest_rgb_time]),
            'depth_rel_path': depth_rel_path
        })

print(f"Matched {len(matched_pairs)} pairs")

Matched 1064 pairs


In [ ]:
import shutil

sync_rgb = "vision_workspace/synced/rgb_raw"
sync_depth = "vision_workspace/synced/depth"
os.makedirs(sync_rgb, exist_ok=True)
os.makedirs(sync_depth, exist_ok=True)

for pair in matched_pairs:
    rgb_name = pair['rgb_filename']
    depth_name = os.path.basename(pair['depth_rel_path'])
    shutil.copy(f"vision_workspace/pipeline_v1_raw/rgb/{rgb_name}", f"{sync_rgb}/{rgb_name}")
    shutil.copy(f"vision_workspace/ground_truth_depth/{depth_name}", f"{sync_depth}/{rgb_name}")

print(f"Synced RGB: {len(os.listdir(sync_rgb))}, Synced depth: {len(os.listdir(sync_depth))}")

Synced RGB: 1057, Synced depth: 1057


In [ ]:
dest = f"{BASE}/{SEQUENCE}/synced_data"
os.makedirs(dest, exist_ok=True)
shutil.copytree("vision_workspace/synced", dest, dirs_exist_ok=True)
print(f"Saved to {dest}")

Saved to /content/drive/MyDrive/vision_lab/structure_texture/synced_data
